# SPARQL Explorer for Chem DCAT-AP

This notebook illustrates how to use the SPARQL Explorer for Chem DCAT-AP to query chemical datasets.

Here are some example queries you can run:

Give me:

- all datasets D about a substance with compound C
- all datasets D generated by Activity A being of type T that evaluated a substance with compound C
- all datasets D generated by Activity A being of type T that evaluated a substance with compound C with Bruker Spectrometer set to X
- all datasets D about chemical reaction R that had product P
- all datasets D about chemical reaction R that had a product with compound C (and similar questions for starting materials, solvents and catalysts)


In [ ]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, Literal, URIRef
from rdflib.namespace import XSD, DC, DCTERMS, FOAF, SKOS
from rdflib.plugins.sparql import prepareQuery
from linkml_runtime.dumpers import rdflib_dumper
from pyshacl import validate
from linkml.generators.shaclgenerator import ShaclGenerator

from chem_dcat_ap.datamodel.chem_dcat_ap_pydantic import ( Distribution, DataService, Catalogue, Dataset, DataGeneratingActivity, 
                                                          DataAnalysis, DatasetSeries, Agent, Activity, AgenticEntity, ChemicalEntity, 
                                                          ChemicalReaction, MaterialEntity )

## 1. Generating some test data sets with the Chem DCAT-AP LinkML data model

Let's model some datasets of different analyses of chemical reactions with the Chem DCAT-AP LinkML data model:

- Dataset D1-D3 about three different chemical reactions (alodol condensation, transamination, Fisher-Tropsch synthesis) with three replicates each, with different starting materials, solvents, catalysts and products.
- Dataset D4-D5 about characterization of three substances with NMR, IR and UV-Vis spectroscopy, with different instruments and settings.



In [ ]:
# Import the dataset generation modules
from .chemical_reaction_datasets import generate_reaction_datasets
from .characterization_datasets import generate_characterization_datasets

# Generate the datasets
reaction_datasets = generate_reaction_datasets()
characterization_datasets = generate_characterization_datasets()

# Combine all datasets
all_datasets = reaction_datasets + characterization_datasets

# Print information about the generated datasets
print(f"Generated {len(reaction_datasets)} chemical reaction datasets")
print(f"Generated {len(characterization_datasets)} characterization datasets")
print(f"Total datasets: {len(all_datasets)}")

# Display basic info for first few datasets
for i, dataset in enumerate(all_datasets[:3]):
    print(f"\nDataset {i+1}: {dataset.title}")
    print(f"  Description: {dataset.description}")
    print(f"  Generated by: {dataset.was_generated_by[0].title if dataset.was_generated_by else 'N/A'}")
    
# Save datasets to JSON files for later use
import json
for i, dataset in enumerate(all_datasets):
    filename = f"dataset_{i+1}.json"
    with open(filename, 'w') as f:
        json.dump(dataset.model_dump(), f, indent=2)
    print(f"Saved {filename}")

## 2. Converting the data to RDF and loading it into an RDF graph /  SPARQL endpoint

In [ ]:
g = rdflib_dumper.as_graph(mol, schemaview="molecule_schema.yaml")

# print(g.serialize(format='turtle').decode('utf-8'))

## 3. Running SPARQL queries against the graph / endpoint

In [ ]:
prefixes = {
    'dcat': 'http://www.w3.org/ns/dcat#',
    'dcterms': 'http://purl.org/dc/terms/',
    'foaf': 'http://xmlns.com/foaf/0.1/',
    'skos': 'http://www.w3.org/2004/02/skos/core#',
    'xsd': 'http://www.w3.org/2001/XMLSchema#'
}

prefix_str = "\n".join([f"PREFIX {prefix}: <{uri}>" for prefix, uri in prefixes.items()])


In [ ]:
# q1: Give me all datasets D about a substance with compound C

query1 = prepareQuery(f"""

{prefix_str}

SELECT ?dataset ?title
WHERE {{
    ?dataset a dcat:Dataset ;
             dcterms:title ?title ;
             chem:about ?substance .
    ?substance chem:hasCompound ?compound .
    FILTER(?compound = "C")
}}""", initNs=prefixes)

In [ ]:
# perform the query and print results

results = g.query(query1)
for row in results:
    print(f"Dataset: {row.dataset}, Title: {row.title}")

In [ ]:
# q2 Give me all datasets D about a substance with compound C that were generated by an activity of type T

query2 = prepareQuery(f"""
{prefix_str}
SELECT ?dataset ?title
WHERE {{
    ?dataset a dcat:Dataset ;
             dcterms:title ?title ;
             chem:about ?substance ;
             dcterms:wasGeneratedBy ?activity .
    ?substance chem:hasCompound ?compound .
    ?activity a ?activityType .
    FILTER(?compound = "C" && ?activityType = T)
}}""", initNs=prefixes)




In [ ]:
results = g.query(query2)
for row in results:
    print(f"Dataset: {row.dataset}, Title: {row.title}")

## 4. Validate the RDF graph with SHACL

In [ ]:

# 1. Generate SHACL shapes as a string/graph from your schema
shacl_string = ShaclGenerator("molecule_schema.yaml").serialize()

# 2. Validate your data graph (g) against the shapes
conforms, results_graph, results_text = validate(
    data_graph=g, 
    shacl_graph=shacl_string,
    inference='rdfs',  # Recommended to handle class hierarchies
    serialize_results_format='turtle'
)

if conforms:
    print("Validation successful!")
else:
    print(f"Validation failed:\n{results_text}")